# Fine-Tune RT-DETR — v2 Two-Stage Training

Fine-tune [RT-DETR r18vd](https://huggingface.co/PekingU/rtdetr_r18vd) on `kaya-go/moku-v2` using a two-stage approach:

1. **Stage 1 — Synthetic pre-training**: Learn general goban structure from ~1000 synthetic images (LR=1e-4, ~30 epochs)
2. **Stage 2 — Real fine-tuning**: Adapt to real-world domain on ~320 corrected real images (HP grid search via `scripts/launch_grid_r*.sh`)

Training runs on HF Jobs (A10G GPU). Results tracked via [W&B](https://wandb.ai/hadim/moku).
Best weights saved as W&B artifacts. Use `20_Analyze_Runs.ipynb` to analyze results, `30_Publish_Model.ipynb` to push to HF Hub.

In [ ]:
%load_ext autoreload
%autoreload 2

from datasets import load_dataset
from transformers import Trainer, TrainingArguments

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.training import (
    collate_fn,
    load_image_processor,
    load_model,
    make_eval_transform,
    make_train_transform,
)

HF_DATASET_V2 = "kaya-go/moku-v2"
HF_MODEL_V2 = "kaya-go/moku-v2"

## Load Datasets

Load both configs from `kaya-go/moku-v2`: synthetic (stage 1) and real (stage 2).

In [ ]:
synthetic_dataset = load_dataset(HF_DATASET_V2, "synthetic")
real_dataset = load_dataset(HF_DATASET_V2, "real")

print("Synthetic:", synthetic_dataset)
print("\nReal:", real_dataset)
print(f"\nCategories: {CATEGORIES}")
print(f"Labels: {ID_TO_CATEGORY}")

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})

Categories: {'black_stone': 0, 'white_stone': 1, 'board_corner': 2}
Labels: {0: 'black_stone', 1: 'white_stone', 2: 'board_corner'}


## Load Model & Image Processor

Load RT-DETR r18vd pre-trained on COCO. Classification head re-initialized for 3 categories.

In [3]:
image_processor = load_image_processor()
model = load_model()

# Print trainable parameter count
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r18vd
Key                                        | Status   |                                                                                        
-------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.enc_score_head.bias                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.denoising_class_embed.weight         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([4, 256])
model.enc_score_head.weight                | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([3, 256])
model.decoder.class_embed.{0, 1, 2}.weight | MISMATCH | Reinit due to si

Total parameters: 20,075,740
Trainable parameters: 20,075,740


## Stage 1 — Synthetic Pre-training (local test)

Quick local test on CPU to verify the pipeline. For full training, use HF Jobs (see below).

In [ ]:
image_processor = load_image_processor()

# Apply transforms to synthetic data
synthetic_dataset["train"].set_transform(make_train_transform(image_processor))
synthetic_dataset["validation"].set_transform(make_eval_transform(image_processor))

# Verify a sample
sample = synthetic_dataset["train"][0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"labels keys: {sample['labels'].keys()}")
print(f"num objects: {len(sample['labels']['class_labels'])}")

pixel_values shape: torch.Size([3, 640, 640])
labels keys: KeysView({'size': tensor([640, 640]), 'image_id': tensor([0]), 'class_labels': tensor([2, 2, 2, 2]), 'boxes': tensor([[0.2164, 0.2188, 0.0266, 0.0344],
        [0.7641, 0.2266, 0.0250, 0.0312],
        [0.1477, 0.9789, 0.0422, 0.0422],
        [0.8250, 0.9809, 0.0344, 0.0367]]), 'area': tensor([374., 320., 729., 517.]), 'iscrowd': tensor([0, 0, 0, 0]), 'orig_size': tensor([640, 640])})
num objects: 4


In [ ]:
# Local test: 2 epochs on CPU just to verify pipeline
model = load_model()

training_args = TrainingArguments(
    output_dir="runs/v2_stage1_test",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    weight_decay=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    use_cpu=True,
    report_to="none",
    run_name="v2_stage1_test",
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=synthetic_dataset["train"],
    eval_dataset=synthetic_dataset["validation"],
)

trainer.train()
print("Stage 1 local test complete.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Output directory: runs/baseline
Epochs: 50
LR: 0.0001
Batch size: 4


## Launch Training on HF Jobs

Full training runs on HF Jobs with A10G GPU. Use `scripts/train.py` which supports:
- `--stage 1` for synthetic pre-training
- `--stage 2` for real fine-tuning from stage 1 checkpoint
- `--resume` to resume an interrupted run
- Best model automatically saved as W&B artifact (by mAP@50:95)

### Stage 1: Synthetic Pre-training

```bash
hf jobs uv run \
    --detach --flavor a10g-small --timeout 3h \
    --secrets HF_TOKEN --secrets-file .env \
    scripts/train.py \
    --stage 1 --run-name v2_stage1 --num-epochs 30 --push-to-hub
```

### Stage 2: HP Grid Search

Use the launch grid script for a batch of runs:

```bash
bash scripts/launch_grid_r4.sh        # Launch all runs in round 4
bash scripts/launch_grid_r4.sh A      # Launch group A only
```

Or single run:

```bash
hf jobs uv run \
    --detach --flavor a10g-large --timeout 3h \
    --secrets HF_TOKEN --secrets-file .env \
    scripts/train.py \
    --stage 2 --round r5 --run-name r5_lr1e-3_cos200 \
    --lr 1e-3 --lr-scheduler cosine --num-epochs 200 --batch-size 16
```

In [ ]:
# Monitor running jobs
import subprocess
result = subprocess.run(["hf", "jobs", "ps"], capture_output=True, text=True)
print(result.stdout)